# Лабораторная работа №3. Ethernet: как выглядит кадр

В Lab 002 мы смотрели на состояние Linux-host. Теперь приблизимся к самому L2-сегменту и разберём **реальный Ethernet II frame**, который выходит из `pc1:eth1`.

Наша цель — не запомнить картинку header, а связать три вещи: **neighbor mapping → destination MAC → фактическая доставка кадра**.

## Схема стенда

![Один Ethernet-сегмент](assets/topology.svg)

MAC на `eth1` фиксированы стендом, чтобы capture был предсказуемым. IPv4 вы добавите сами. `eth0` — management plane; его не меняем.

## 1. Сначала увидим L2 identity интерфейсов

На `pc1`:

```bash
ip -br link show eth1
```

На `pc2`:

```bash
ip -br link show eth1
```

У `pc1:eth1` должен быть MAC `02:00:00:00:03:01`, у `pc2:eth1` — `02:00:00:00:03:02`.

### Подготовим L3 только как источник полезного трафика

На `pc1`:

```bash
ip -4 addr flush dev eth1
ip addr add 198.51.100.10/24 dev eth1
ip neigh flush dev eth1
ip neigh replace 198.51.100.20 lladdr 02:00:00:00:03:02 nud permanent dev eth1
ip -br addr show eth1
ip neigh show dev eth1
```

На `pc2`:

```bash
ip -4 addr flush dev eth1
ip addr add 198.51.100.20/24 dev eth1
ip neigh flush dev eth1
ip neigh replace 198.51.100.10 lladdr 02:00:00:00:03:01 nud permanent dev eth1
ip -br addr show eth1
ip neigh show dev eth1
```

Neighbor mapping здесь задаём вручную **только чтобы убрать ARP из эксперимента**. Как Linux узнаёт этот MAC автоматически, подробно разберём в Lab 005.

### Q1. Когда `pc1` отправит ICMP Echo Request к `198.51.100.20`, какой MAC должен быть source, какой destination и почему destination берётся не из IPv4 destination напрямую?

`введите ваш ответ`

<details>
<summary>Проверить ответ с ИИ (необязательно)</summary>

**AI-REVIEW-PROMPT / REVIEW-ONLY**

Проверь мой ответ на вопрос Q1, который находится выше.  
Не отвечай на исходный вопрос и не давай эталонный ответ.  
Проверь логику рассуждения, укажи на возможный пропущенный шаг и задай 1–2 наводящих вопроса.  
Не переписывай мой ответ и не давай готовую CLI-команду следующего задания.

</details>

## 2. Что находится в начале Ethernet II frame

![Ethernet II header](assets/frame.svg)

Первые 14 байт — destination MAC, source MAC и EtherType. После них начинается payload. FCS обычно не виден в capture, сделанном внутри Linux.

### Q2. До запуска capture предскажите первые 14 байт по смыслу: какие 6 байт идут первыми, какие 6 следующими и что должны означать байты 12–13 для кадра с IPv4/ICMP payload?

`введите ваш ответ`

<details>
<summary>Проверить ответ с ИИ (необязательно)</summary>

**AI-REVIEW-PROMPT / REVIEW-ONLY**

Проверь мой ответ на вопрос Q2, который находится выше.  
Не отвечай на исходный вопрос и не давай эталонный ответ.  
Проверь логику рассуждения, укажи на возможный пропущенный шаг и задай 1–2 наводящих вопроса.  
Не переписывай мой ответ и не давай готовую CLI-команду следующего задания.

</details>

### Получим настоящий кадр

Запустите следующую cell. Пока идёт capture, в terminal `pc1` выполните:

```bash
ping -c 3 198.51.100.20
```

Фильтр оставляем `icmp`: благодаря permanent neighbor entry ARP здесь не нужен, и в capture остаётся именно интересующий нас unicast traffic.

In [ ]:
%%capture_traffic pc1:eth1 --filter "icmp" --timeout 10 --packets 30 --save captures/ethernet-good.pcap
import time
print("Сейчас выполните в terminal pc1: ping -c 3 198.51.100.20")
time.sleep(8)


### Наблюдение: таблица, protocol tree и hex dump

Ниже остаётся обычный notebook-output из того же PCAP. Он виден и после статического HTML/PDF-экспорта.

In [ ]:
from cms_labs_jupyter.traffic_capture import PcapViewer
from IPython.display import display

viewer = PcapViewer("captures/ethernet-good.pcap", "icmp", limit=20)
frames = viewer.packets()
display(frames)

icmp = frames[frames["Protocol"].astype(str).str.contains("ICMP", case=False, na=False)]
if not icmp.empty:
    frame_number = int(icmp.iloc[0]["№"])
    print(f"\nProtocol tree · frame {frame_number}\n")
    print("\n".join(viewer.packet(frame_number, "protocols").splitlines()[:55]))
    print(f"\nHex dump · frame {frame_number}\n")
    print("\n".join(viewer.packet(frame_number, "hex").splitlines()[:12]))
else:
    print("ICMP-кадр не найден: повторите capture и ping.")


### Что должно совпасть

В Echo Request source MAC относится к `pc1:eth1`, destination MAC — к `pc2:eth1`, а EtherType сообщает, что payload содержит IPv4. Сам IPv4 destination при этом остаётся `198.51.100.20`.

То есть IP отвечает **куда end-to-end**, а Ethernet header — **кому доставить frame на текущем L2-сегменте**.

## Один и тот же процесс как движение кадра

![Как Ethernet frame проходит сегмент](assets/frame-flight.svg)

Анимация показывает не новый протокол, а причинную цепочку: neighbor mapping даёт destination MAC → header описывает payload → frame проходит link → receiver принимает его, если destination MAC подходит.

### Q3. Почему EtherType нужен даже тогда, когда source и destination MAC уже известны? Что receiver не смог бы однозначно понять без этого поля?

`введите ваш ответ`

<details>
<summary>Проверить ответ с ИИ (необязательно)</summary>

**AI-REVIEW-PROMPT / REVIEW-ONLY**

Проверь мой ответ на вопрос Q3, который находится выше.  
Не отвечай на исходный вопрос и не давай эталонный ответ.  
Проверь логику рассуждения, укажи на возможный пропущенный шаг и задай 1–2 наводящих вопроса.  
Не переписывай мой ответ и не давай готовую CLI-команду следующего задания.

</details>

## 3. CHECK рабочего Ethernet-состояния

На `pc1` перед CHECK:

```bash
ip -br link show eth1
ip -br addr show eth1
ip neigh show 198.51.100.20 dev eth1
ip route get 198.51.100.20
ping -c 2 198.51.100.20
```

Запустите **CHECK**. Он проверяет фиксированные MAC обоих интерфейсов, IPv4, правильные neighbor mappings и двустороннюю связность.

## 4. Controlled failure: IP и route оставим правильными, MAC mapping подменим

Сейчас изменим ровно одну вещь: скажем `pc1`, будто IP `198.51.100.20` соответствует **неправильному** MAC `02:00:00:00:03:ff`.

### Q4. После подмены neighbor mapping что должно остаться прежним в `ip route get` и IPv4-конфигурации? Какой destination MAC вы ожидаете увидеть у исходящих ICMP frames и что произойдёт с ping?

`введите ваш ответ`

<details>
<summary>Проверить ответ с ИИ (необязательно)</summary>

**AI-REVIEW-PROMPT / REVIEW-ONLY**

Проверь мой ответ на вопрос Q4, который находится выше.  
Не отвечай на исходный вопрос и не давай эталонный ответ.  
Проверь логику рассуждения, укажи на возможный пропущенный шаг и задай 1–2 наводящих вопроса.  
Не переписывай мой ответ и не давай готовую CLI-команду следующего задания.

</details>

На `pc1`:

```bash
ip neigh replace 198.51.100.20 lladdr 02:00:00:00:03:ff nud permanent dev eth1
ip neigh show 198.51.100.20 dev eth1
ip route get 198.51.100.20
```

Теперь запустите **CHECK**. Не исправляйте сразу. Обратите внимание: MAC самого `pc1`, его IPv4 и route не должны исчезнуть — сломан именно mapping следующего L2-соседа.

### Посмотрим неправильный кадр на проводе

Запустите capture и во время него выполните на `pc1`:

```bash
ping -c 2 -W 1 198.51.100.20
```

Даже неуспешный ping может оставить доказательство: исходящий Echo Request способен появиться в capture с неправильным destination MAC.

In [ ]:
%%capture_traffic pc1:eth1 --filter "icmp" --timeout 8 --packets 20 --save captures/ethernet-wrong-mac.pcap
import time
print("Сейчас выполните в terminal pc1: ping -c 2 -W 1 198.51.100.20")
time.sleep(6)


In [ ]:
from cms_labs_jupyter.traffic_capture import PcapViewer
from IPython.display import display

wrong = PcapViewer("captures/ethernet-wrong-mac.pcap", "icmp", limit=20)
wrong_frames = wrong.packets()
display(wrong_frames)

if not wrong_frames.empty:
    frame_number = int(wrong_frames.iloc[0]["№"])
    print(f"\nProtocol tree · wrong-MAC frame {frame_number}\n")
    print("\n".join(wrong.packet(frame_number, "protocols").splitlines()[:45]))
    print(f"\nHex dump · wrong-MAC frame {frame_number}\n")
    print("\n".join(wrong.packet(frame_number, "hex").splitlines()[:10]))
else:
    print("ICMP не найден: повторите capture и ping во время захвата.")


### Q5. Если capture показывает исходящий ICMP Echo Request, `ip route get` по-прежнему выбирает `eth1`, но Echo Reply не приходит, какое наблюдение указывает на L2-причину? Объясните цепочку причины и симптомов.

`введите ваш ответ`

<details>
<summary>Проверить ответ с ИИ (необязательно)</summary>

**AI-REVIEW-PROMPT / REVIEW-ONLY**

Проверь мой ответ на вопрос Q5, который находится выше.  
Не отвечай на исходный вопрос и не давай эталонный ответ.  
Проверь логику рассуждения, укажи на возможный пропущенный шаг и задай 1–2 наводящих вопроса.  
Не переписывай мой ответ и не давай готовую CLI-команду следующего задания.

</details>

### Диагностика

Сравните три факта на `pc1`:

```bash
ip -br addr show eth1
ip route get 198.51.100.20
ip neigh show 198.51.100.20 dev eth1
```

IPv4 и route всё ещё ведут к локальному `eth1`. Отличается neighbor mapping: именно он дал кадру неверный destination MAC. Поэтому наличие IP packet и правильного route ещё не гарантирует успешную L2-доставку.

## 5. Восстановление и FINAL CHECK

На `pc1` верните правильный mapping:

```bash
ip neigh replace 198.51.100.20 lladdr 02:00:00:00:03:02 nud permanent dev eth1
ip neigh show 198.51.100.20 dev eth1
ping -c 3 198.51.100.20
```

Запустите **CHECK** ещё раз. Все проверки должны снова стать зелёными.

## Что нужно унести из Lab 003

- Ethernet II header начинается с destination MAC, затем source MAC и EtherType;
- destination MAC относится к доставке на текущем L2-сегменте;
- source MAC берётся с outgoing interface;
- EtherType сообщает, как интерпретировать payload;
- IPv4 destination и Ethernet destination решают разные задачи;
- правильный IP route не компенсирует неправильный destination MAC;
- capture позволяет увидеть не только успешный обмен, но и причину неуспешной доставки.

В Lab 004 добавим switch и посмотрим, как он учится отправлять такие кадры на правильный порт.